# Human endometrium: decidual vs fibroblast nuclei, pseudobulk DESeq2

Decidual cells and endometrial fibroblasts of our human endometrium (Slide-tags; donor mck5, menstrual phase, and
donor mck6, secretory phase), labeled by obs `cell_type`. The raw counts (`layers/counts`) are summed into one
pseudobulk per donor and arm, 4 in all, and the two arms are compared with PyDESeq2: design `~ donor + group`, Wald
test of decidual vs fibroblast, so a positive log2 fold change means higher in decidual nuclei. Donor and phase are
the same factor, which leaves 1 residual degree of freedom: fold changes and per-donor directions are informative,
but the p-values are not evidence that a difference replicates across women.

This is fit `B_new_celltype_release`, the primary human table, of the code of record
(`DE_IVD_code_2026-10-31/3_human_slidetags/`: `human_pseudobulk_job.py`, `fit_human_de.py`,
`build_human_de_tables.py`), fitted on the cluster on 29 October 2026: 20,211 genes tested, 3,198 called DE
(1,867 up, 1,331 down). The other fits there (the `coarse_refined` labels, one pseudobulk per Slide-tags section, the
old notebook's settings) are sensitivity and regression checks and are not repeated here.

- **Input:** `H5AD`, the human fibroblast/decidual object (obs `cell_type`, `donor`, `timepoint`; `layers/counts`).
- **Outputs, in `OUT_DIR`:** `human_decidual_vs_fibroblast_all_genes.csv` (one row per gene in the object, with the
  table of record's columns), `pseudobulks.csv`, `pseudobulk_counts.csv.gz`, `pseudobulk_detected.csv.gz` and
  `check_against_record.csv`.
- **Runtime and memory:** 2.5 to 6 minutes with 8 CPUs, most of it importing the libraries and reading the object
  (the fit itself takes under a minute); peak memory about 3 GB.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them. Relative paths start from this notebook's folder.

# Input of record: the object the fit of record was made on.
H5AD = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/deliverables/2026_09_15/Cevrim_Human_Only_Fib_Dec_Cells.h5ad"

# The table of record, compared with this run at the end; None skips that comparison.
RECORD = "record/human_decidual_vs_fibroblast_all_genes.csv.gz"

OUT_DIR = None  # None writes nothing
FORCE = False  # True lets this run replace the outputs already in OUT_DIR
N_CPUS = 8  # PyDESeq2 workers; 1 and 8 give identical results

In [ ]:
import io
import platform
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pydeseq2
import scipy.sparse as sp
from anndata.io import read_elem
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

# The two-donor p-values are sensitive to the numerical environment (outside the cluster, with NumPy 2.4.6, the same
# code called 3,195 genes instead of 3,198), so the executed notebook records the versions.
print(
    f"Python {platform.python_version()}, NumPy {np.__version__}, "
    f"pandas {pd.__version__}, PyDESeq2 {pydeseq2.__version__}"
)

## Settings

The settings of the fit of record.

In [ ]:
# Settings of the fit of record (fit_human_de.py, "release" settings, shared with the X-Mens and Ulrich fits).
LABEL_COLUMN = "cell_type"  # the post-relabel annotation (chosen 25 October 2026); coarse_refined was a sensitivity fit
ARM = {"decidual_cell": "decidual", "endometrial_fibroblast": "fibroblast"}
MIN_NUCLEI = 20  # per pseudobulk: the Ulrich release's cell floor; the smallest pseudobulk here has 5,737
MIN_CPM = 1.0  # gene filter (see Fit)
MAX_PADJ = 0.05  # a gene is called DE at adjusted p below this
MIN_ABS_LOG2FC = 1  # and unshrunk |log2 fold change| above this
PSEUDOCOUNT_CPM = 0.5  # for the per-donor log2 CPM ratios

# Fail now, not after the fit, if this run would replace earlier outputs.
if OUT_DIR is not None and Path(OUT_DIR).exists() and not FORCE:
    raise FileExistsError(f"{OUT_DIR} exists; set FORCE = True to replace its outputs")

## Load

Reads obs, var and the raw counts (`layers/counts`) only. `X` holds normalized values and is not needed, so the file
is opened with h5py rather than loaded whole. Every nucleus in the object is a decidual cell or an endometrial
fibroblast.

In [ ]:
with h5py.File(H5AD, "r") as f:
    obs = read_elem(f["obs"])
    var = read_elem(f["var"])
    counts = sp.csr_matrix(read_elem(f["layers/counts"]))

# The sums below need UMI counts; a normalized layer would change every result without raising an error.
assert np.array_equal(counts.data, np.round(counts.data)), "layers/counts holds non-integer values"
counts = counts.astype(np.int64)

# var's index holds gene symbols; the fit and the table are keyed by Ensembl ID.
gene_ids = var["gene_ids"].astype(str).to_numpy()
symbols = var.index.astype(str).to_numpy()
assert len(set(gene_ids)) == len(gene_ids), "Ensembl gene IDs are not unique"

# The release objects in final_objects/ have no donor column. Their sample names end in the donor and a section
# letter (menstrual_mck5a is donor mck5).
if "donor" not in obs.columns:
    obs["donor"] = obs["sample"].astype(str).str.extract(r"_(mck\d+)[a-z]$", expand=False)
    print("obs has no donor column; donor taken from the sample names")

arm = obs[LABEL_COLUMN].astype(str).map(ARM)
assert arm.notna().all(), f"{LABEL_COLUMN} values outside ARM: {sorted(obs[LABEL_COLUMN].astype(str).unique())}"

print(f"{counts.shape[0]:,} nuclei x {counts.shape[1]:,} genes")
pd.crosstab([obs["donor"], obs["timepoint"]], arm)

## Pseudobulks

One pseudobulk per donor and arm: the UMIs of its nuclei summed per gene, and the number of its nuclei with at least
one UMI of each gene (for the detection columns of the table). They are built in the order of the fit of record
(donors sorted, fibroblast before decidual), so PyDESeq2 sees its samples in the same order.

In [ ]:
is_detected = (counts > 0).astype(np.int64)  # 1 where a nucleus has at least one UMI of a gene
umis_per_nucleus = np.asarray(counts.sum(axis=1)).ravel()
donor_of_nucleus = obs["donor"].astype(str)

descriptions, summed_counts, detecting_nuclei = [], [], []
for donor in sorted(donor_of_nucleus.unique()):
    for arm_name in ("fibroblast", "decidual"):
        in_pseudobulk = ((donor_of_nucleus == donor) & (arm == arm_name)).to_numpy()
        nuclei = np.flatnonzero(in_pseudobulk)

        # Each donor was sampled in one phase.
        phases = obs.loc[in_pseudobulk, "timepoint"].astype(str).unique()
        assert len(phases) == 1, (donor, phases)

        descriptions.append(
            {
                "id": f"{donor}_{arm_name}",
                "donor": donor,
                "timepoint": phases[0],
                "arm": arm_name,
                "n_cells": len(nuclei),
                "total_counts": int(umis_per_nucleus[nuclei].sum()),
            }
        )
        summed_counts.append(np.asarray(counts[nuclei].sum(axis=0)).ravel())
        detecting_nuclei.append(np.asarray(is_detected[nuclei].sum(axis=0)).ravel())

pseudobulks = pd.DataFrame(descriptions).set_index("id")
pseudobulk_counts = pd.DataFrame(np.vstack(summed_counts), index=pseudobulks.index, columns=gene_ids)
pseudobulk_detected = pd.DataFrame(np.vstack(detecting_nuclei), index=pseudobulks.index, columns=gene_ids)
pseudobulks

## Fit

**Gene filter.** A gene is tested if its CPM is at least 1 in at least half (rounded up) of the decidual pseudobulks
or of the fibroblast pseudobulks: with two donors, in either donor's pseudobulk of either arm. Each arm is checked on
its own, so a gene switched on in only one arm is kept. This is the rule of the X-Mens and Ulrich fits
(`arm_aware_eligible()` in `ulrich_ivd_pipeline.py`).

In [ ]:
# The 20-nucleus floor is shared with the X-Mens and Ulrich fits; it excludes no pseudobulk here.
in_fit = pseudobulks.index[pseudobulks["n_cells"] >= MIN_NUCLEI]
print(f"{len(in_fit)} of {len(pseudobulks)} pseudobulks have at least {MIN_NUCLEI} nuclei")

# CPM over all genes of a pseudobulk, before any filtering; also used for the per-donor ratios in Results.
cpm = pseudobulk_counts.div(pseudobulk_counts.sum(axis=1), axis=0) * 1e6

fit_cpm = cpm.loc[in_fit]
is_decidual = (pseudobulks.loc[in_fit, "arm"] == "decidual").to_numpy()
needed_in_decidual = (int(is_decidual.sum()) + 1) // 2
needed_in_fibroblast = (int((~is_decidual).sum()) + 1) // 2
passes_in_decidual = fit_cpm.loc[is_decidual].ge(MIN_CPM).sum(axis=0).ge(needed_in_decidual)
passes_in_fibroblast = fit_cpm.loc[~is_decidual].ge(MIN_CPM).sum(axis=0).ge(needed_in_fibroblast)
is_tested = passes_in_decidual | passes_in_fibroblast
print(f"{int(is_tested.sum()):,} of {len(is_tested):,} genes pass the filter")

**Model.** PyDESeq2 with `~ donor + group`, Cook's outlier refit on and independent filtering off, so every gene that
passed the filter is tested and the Benjamini-Hochberg adjustment runs over all of them. `group` stays a plain string
column, as in the fit of record: its reference level is then the alphabetically first, decidual, and the contrast
sets the direction. PyDESeq2 warns that with fewer than 3 residual degrees of freedom the spread of the log
dispersions, which sets the width of their prior, is poorly estimated; that is the 1-residual-degree-of-freedom
caveat above.

In [ ]:
metadata = pd.DataFrame(
    {
        "donor": pseudobulks.loc[in_fit, "donor"].astype(str).to_numpy(),
        "group": pseudobulks.loc[in_fit, "arm"].astype(str).to_numpy(),
    },
    index=in_fit,
)

inference = DefaultInference(n_cpus=N_CPUS)
dds = DeseqDataSet(
    counts=pseudobulk_counts.loc[in_fit, is_tested.to_numpy()].astype(int),
    metadata=metadata,
    design="~ donor + group",
    refit_cooks=True,
    inference=inference,
    quiet=True,
)
dds.deseq2()

# The design needs full rank: group must not be confounded with donor.
design_matrix = dds.obsm["design_matrix"]
rank = int(np.linalg.matrix_rank(design_matrix.to_numpy(dtype=float)))
assert rank == design_matrix.shape[1], (rank, list(design_matrix.columns))
residual_df = len(metadata) - rank
print(f"residual degrees of freedom: {residual_df}")
design_matrix

In [ ]:
stats = DeseqStats(
    dds,
    contrast=["group", "decidual", "fibroblast"],
    independent_filter=False,
    inference=inference,
    quiet=True,
)
stats.summary()

results = stats.results_df[["baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj"]].copy()
results["estimable"] = np.isfinite(results["log2FoldChange"]) & np.isfinite(results["pvalue"])

# A missing adjusted p-value would count as not significant; Cook's filtering leaves none here.
padj_or_1 = results["padj"].where(np.isfinite(results["padj"]), 1.0)
results["call"] = (
    results["estimable"]
    & padj_or_1.lt(MAX_PADJ)
    & results["log2FoldChange"].abs().gt(MIN_ABS_LOG2FC)
)
results["direction"] = np.where(
    results["log2FoldChange"] > 0,
    "up",
    np.where(results["log2FoldChange"] < 0, "down", ""),
)
print(f"{int(results['padj'].isna().sum())} tested genes have no adjusted p-value")

## Results

One row per gene in the object, with the columns of the X-Mens release's tables. Genes that did not pass the filter
have `production_eligible` False and empty statistics. `direction` is the sign of the fold change of every tested
gene, called or not. `log2_cpm_ratio_<donor>_<phase>` is log2((CPM decidual + 0.5) / (CPM fibroblast + 0.5)) within
one donor, filled in for tested genes only; `pct_detected_<donor>_<phase>_<arm>` is the percent of the pseudobulk's
nuclei with at least one UMI of the gene. Both are rounded to 3 decimals, as in the table of record.

In [ ]:
statistics = results.rename(
    columns={"log2FoldChange": "log2FoldChange_unshrunk", "call": "significant_fdr05_lfc1"}
)
for donor, of_donor in pseudobulks.groupby("donor"):
    decidual = of_donor.index[of_donor["arm"] == "decidual"][0]
    fibroblast = of_donor.index[of_donor["arm"] == "fibroblast"][0]
    phase = of_donor.loc[decidual, "timepoint"]
    ratio = np.log2((cpm.loc[decidual] + PSEUDOCOUNT_CPM) / (cpm.loc[fibroblast] + PSEUDOCOUNT_CPM))
    statistics[f"log2_cpm_ratio_{donor}_{phase}"] = ratio.reindex(statistics.index).round(3)

table = pd.DataFrame(
    {
        "ensembl_gene_id": gene_ids,
        "gene_symbol": symbols,
        "production_eligible": is_tested.to_numpy(),
    }
)
table = table.merge(statistics, left_on="ensembl_gene_id", right_index=True, how="left")

# Untested genes are neither estimable nor called, and their padj counts as 1 for the call.
table["significant_fdr05_lfc1"] = table["significant_fdr05_lfc1"].eq(True)
table["estimable"] = table["estimable"].eq(True)
table["padj_for_call"] = table["padj"].where(np.isfinite(table["padj"]), 1.0)

for pseudobulk_id, pseudobulk in pseudobulks.iterrows():
    percent = 100 * pseudobulk_detected.loc[pseudobulk_id] / pseudobulk["n_cells"]
    column = f"pct_detected_{pseudobulk['donor']}_{pseudobulk['timepoint']}_{pseudobulk['arm']}"
    table[column] = percent.round(3).to_numpy()

# Provenance, in the release's columns; the object is named relative to the lab folder, as in the record.
table["source_id"] = f"human_decidual_vs_fibroblast_{LABEL_COLUMN}_by_donor"
table["label_column"] = LABEL_COLUMN
table["positive_lfc_meaning"] = "decidual_minus_fibroblast"
table["analysis_method"] = "pydeseq2_wald_fixed_block"
table["object"] = str(H5AD).removeprefix("/n/lab_storage/eddy_lab/users/nhilgert/") + " (layers/counts)"
table.head()

In [ ]:
is_called = table["significant_fdr05_lfc1"]
n_tested = int(table["production_eligible"].sum())
n_up = int((is_called & (table["log2FoldChange_unshrunk"] > 0)).sum())
n_down = int((is_called & (table["log2FoldChange_unshrunk"] < 0)).sum())
print(f"{n_tested:,} genes tested; {n_up + n_down:,} called DE ({n_up:,} up and {n_down:,} down in decidual nuclei)")

# Known decidualization genes, for orientation; README_human_DE_rerun.md tabulates the same ones.
markers = ["PRL", "IGFBP1", "LEFTY2", "FOXO1", "BMP2", "ZBTB16", "HAND2"]
ratio_columns = [column for column in table.columns if column.startswith("log2_cpm_ratio_")]
shown = ["log2FoldChange_unshrunk", "lfcSE", "padj", "significant_fdr05_lfc1", *ratio_columns]
table.set_index("gene_symbol").loc[markers, shown]

## Check against the record

Compares the table with the table of record: the same genes, identical values in every column the two share, and
identical DE calls. Values are compared as read back from CSV with round-trip precision and no tolerance; an empty
cell equals an empty cell. Skipped when `RECORD` is None. A difference stops the notebook before anything is saved.

In [ ]:
check = None
check_result = "skipped (RECORD is None)"
if RECORD is not None:
    # The table takes the same text round trip as the record, so both are compared exactly as a CSV reader sees them.
    ours = pd.read_csv(io.StringIO(table.to_csv(index=False)), float_precision="round_trip")
    record = pd.read_csv(RECORD, float_precision="round_trip")

    n_record_genes = len(record)
    same_genes = ours["ensembl_gene_id"].tolist() == record["ensembl_gene_id"].tolist()
    ours = ours.set_index("ensembl_gene_id")
    record = record.set_index("ensembl_gene_id").reindex(ours.index)

    rows = []
    for column in [column for column in ours.columns if column in record.columns]:
        ours_values, record_values = ours[column], record[column]
        is_same = (ours_values == record_values) | (ours_values.isna() & record_values.isna())
        is_numeric = pd.api.types.is_float_dtype(ours_values) and pd.api.types.is_float_dtype(record_values)
        rows.append(
            {
                "column": column,
                "n_different": int((~is_same).sum()),
                "max_abs_difference": (ours_values - record_values).abs().max() if is_numeric else np.nan,
            }
        )
    check = pd.DataFrame(rows)

    calls = set(ours.index[ours["significant_fdr05_lfc1"].eq(True)])
    record_calls = set(record.index[record["significant_fdr05_lfc1"].eq(True)])
    missing_columns = sorted(set(record.columns) - set(ours.columns))
    differing_columns = check.loc[check["n_different"] > 0, "column"].tolist()

    print(f"genes: {len(ours):,} here, {n_record_genes:,} in the record; same genes in the same order: {same_genes}")
    print(f"columns compared: {len(check) + 1} (gene ID included); with any difference: {differing_columns or 'none'}")
    print(f"record columns not reproduced: {missing_columns or 'none'}")
    print(f"DE calls: {len(calls):,} here, {len(record_calls):,} in the record; identical: {calls == record_calls}")

    passed = same_genes and calls == record_calls and not differing_columns and not missing_columns
    check_result = "identical to the record" if passed else "DIFFERS from the record"
    if not passed:
        print(check[check["n_different"] > 0].to_string(index=False))
        raise AssertionError(
            f"the table differs from the record: columns {differing_columns}, missing {missing_columns}"
        )
print(check_result)

## Save

Writes into `OUT_DIR` through a sibling folder `<OUT_DIR>.partial` that is renamed once every file is written, so a
failed run never leaves a folder that looks complete. Nothing is written when `OUT_DIR` is None.

In [ ]:
if OUT_DIR is None:
    print("OUT_DIR is None; nothing written")
else:
    out_dir = Path(OUT_DIR)
    partial_dir = out_dir.with_name(out_dir.name + ".partial")
    partial_dir.mkdir(parents=True, exist_ok=True)

    table.to_csv(partial_dir / "human_decidual_vs_fibroblast_all_genes.csv", index=False)
    pseudobulks.to_csv(partial_dir / "pseudobulks.csv")
    pseudobulk_counts.to_csv(partial_dir / "pseudobulk_counts.csv.gz")
    pseudobulk_detected.to_csv(partial_dir / "pseudobulk_detected.csv.gz")
    if check is not None:
        check.to_csv(partial_dir / "check_against_record.csv", index=False)

    # With FORCE, files of the same name in OUT_DIR are replaced and nothing else there is touched.
    if out_dir.exists():
        for path in partial_dir.iterdir():
            path.replace(out_dir / path.name)
        partial_dir.rmdir()
    else:
        partial_dir.rename(out_dir)
    print(f"wrote {out_dir}: {', '.join(sorted(path.name for path in out_dir.iterdir()))}")

## Summary

In [ ]:
print(f"pseudobulks: {len(in_fit)} (residual degrees of freedom: {residual_df})")
print(f"genes tested: {n_tested:,}")
print(
    f"genes called DE (padj < {MAX_PADJ}, |log2FC| > {MIN_ABS_LOG2FC}): "
    f"{n_up + n_down:,} ({n_up:,} up, {n_down:,} down)"
)
print(f"check against the record: {check_result}")